<a href="https://colab.research.google.com/github/francji1/01ZLMA/blob/main/code/01ZLMA_ex01_LLM_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW 01 — Solutions: Exponential Family "Big Five" + Bernoulli

## Variance functions of the basic exponential family distributions

For each distribution we:
1. Write the PDF/PMF in exponential-family form $f(y;\theta,\phi) = \exp\!\left[\frac{y\theta - b(\theta)}{a(\phi)} - c(y,\phi)\right]$
2. Identify $\theta$, $b(\theta)$, $a(\phi)$, $c(y,\phi)$
3. Compute $E[Y] = b'(\theta)$, $\mathrm{Var}[Y] = a(\phi)\,b''(\theta)$
4. Derive the variance function $v(\mu) = b''(\theta(\mu))$ so that $\mathrm{Var}[Y] = a(\phi)\,v(\mu)$
5. Identify the canonical link function $g(\mu) = \theta$

Each section ends with a Monte Carlo simulation (100 runs) to numerically verify the theoretical $E[Y]$ and $\mathrm{Var}[Y]$.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n_samples = 10000  # samples per simulation
n_sims = 100       # number of simulations

---

## 1. Normal Distribution $N(\mu, \sigma^2)$

**PDF:**
$$
f(y;\mu,\sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(y-\mu)^2}{2\sigma^2}\right)
$$

### Step 1: Rewrite in exponential-family form

Expand the exponent:
$$
-\frac{(y-\mu)^2}{2\sigma^2} = -\frac{y^2 - 2y\mu + \mu^2}{2\sigma^2} = \frac{y\mu - \frac{\mu^2}{2}}{\sigma^2} - \frac{y^2}{2\sigma^2}
$$

Therefore:
$$
f(y;\mu,\sigma^2) = \exp\!\left[\frac{y\mu - \frac{\mu^2}{2}}{\sigma^2} - \left(\frac{y^2}{2\sigma^2} + \frac{1}{2}\ln(2\pi\sigma^2)\right)\right]
$$

### Step 2: Identify components

| Component | Value |
|-----------|-------|
| Natural parameter | $\theta = \mu$ |
| $b(\theta)$ | $\frac{\theta^2}{2}$ |
| $a(\phi)$ | $\sigma^2$ |
| $c(y,\phi)$ | $\frac{y^2}{2\sigma^2} + \frac{1}{2}\ln(2\pi\sigma^2)$ |

### Step 3: Compute moments

$$
E[Y] = b'(\theta) = \frac{d}{d\theta}\frac{\theta^2}{2} = \theta = \mu
$$

$$
b''(\theta) = \frac{d^2}{d\theta^2}\frac{\theta^2}{2} = 1
$$

$$
\mathrm{Var}[Y] = a(\phi)\,b''(\theta) = \sigma^2 \cdot 1 = \sigma^2
$$

### Step 4: Variance function

$$
v(\mu) = b''(\theta(\mu)) = 1
$$

### Step 5: Canonical link

Since $\theta = \mu$, the canonical link is the **identity**: $g(\mu) = \mu$.

### Verification via MGF

The MGF of $Y \sim N(\mu, \sigma^2)$ is:
$$
M_Y(t) = \exp\!\left(\mu t + \frac{\sigma^2 t^2}{2}\right)
$$
$$
M_Y'(t) = \left(\mu + \sigma^2 t\right) M_Y(t), \quad M_Y'(0) = \mu = E[Y] \;\checkmark
$$
$$
M_Y''(t) = \sigma^2 M_Y(t) + \left(\mu + \sigma^2 t\right)^2 M_Y(t), \quad M_Y''(0) = \sigma^2 + \mu^2
$$
$$
\mathrm{Var}[Y] = M_Y''(0) - (M_Y'(0))^2 = \sigma^2 + \mu^2 - \mu^2 = \sigma^2 \;\checkmark
$$

### Summary

$$
\boxed{E[Y] = \mu, \quad \mathrm{Var}[Y] = \sigma^2, \quad v(\mu) = 1, \quad g(\mu) = \mu \text{ (identity)}}
$$

**Note:** Normal is the only distribution in the "Big Five" where the variance does not depend on $\mu$ (homoscedastic).

In [2]:
# --- Normal: Monte Carlo verification ---
mu_true, sigma2_true = 5.0, 4.0
means, variances = [], []
for _ in range(n_sims):
    y = np.random.normal(loc=mu_true, scale=np.sqrt(sigma2_true), size=n_samples)
    means.append(np.mean(y))
    variances.append(np.var(y, ddof=1))

print("=== Normal N(mu={}, sigma^2={}) ===".format(mu_true, sigma2_true))
print(f"  Theoretical E[Y]   = {mu_true}")
print(f"  Simulated   E[Y]   = {np.mean(means):.4f}  (std over sims: {np.std(means):.4f})")
print(f"  Theoretical Var[Y] = {sigma2_true}")
print(f"  Simulated   Var[Y] = {np.mean(variances):.4f}  (std over sims: {np.std(variances):.4f})")
print(f"  v(mu) = 1  =>  Var[Y] / sigma^2 = {np.mean(variances)/sigma2_true:.4f}  (should be ~1)")

=== Normal N(mu=5.0, sigma^2=4.0) ===
  Theoretical E[Y]   = 5.0
  Simulated   E[Y]   = 4.9968  (std over sims: 0.0192)
  Theoretical Var[Y] = 4.0
  Simulated   Var[Y] = 4.0015  (std over sims: 0.0537)
  v(mu) = 1  =>  Var[Y] / sigma^2 = 1.0004  (should be ~1)


---

## 2. Poisson Distribution $\mathrm{Pois}(\lambda)$

**PMF:**
$$
P(Y = y) = \frac{\lambda^y e^{-\lambda}}{y!}, \quad y = 0, 1, 2, \ldots
$$

### Step 1: Rewrite in exponential-family form

$$
f(y;\lambda) = \frac{\lambda^y e^{-\lambda}}{y!} = \exp\!\bigl[y \ln(\lambda) - \lambda - \ln(y!)\bigr]
$$

Comparing with the general form $\exp\!\left[\frac{y\theta - b(\theta)}{a(\phi)} - c(y,\phi)\right]$ (here $a(\phi)=1$):

### Step 2: Identify components

| Component | Value |
|-----------|-------|
| Natural parameter | $\theta = \ln(\lambda)$, i.e. $\lambda = e^\theta$ |
| $b(\theta)$ | $e^\theta$ |
| $a(\phi)$ | $1$ (known dispersion) |
| $c(y,\phi)$ | $\ln(y!)$ |

### Step 3: Compute moments

$$
E[Y] = b'(\theta) = \frac{d}{d\theta} e^\theta = e^\theta = \lambda
$$

$$
b''(\theta) = \frac{d^2}{d\theta^2} e^\theta = e^\theta = \lambda
$$

$$
\mathrm{Var}[Y] = a(\phi)\,b''(\theta) = 1 \cdot \lambda = \lambda
$$

### Step 4: Variance function

Since $\mu = \lambda = e^\theta$:
$$
v(\mu) = b''(\theta(\mu)) = e^\theta = \mu
$$

### Step 5: Canonical link

Since $\theta = \ln(\lambda) = \ln(\mu)$, the canonical link is the **log**: $g(\mu) = \ln(\mu)$.

### Verification via MGF

The MGF of $Y \sim \mathrm{Pois}(\lambda)$ is:
$$
M_Y(t) = \exp\!\bigl[\lambda(e^t - 1)\bigr]
$$
$$
M_Y'(t) = \lambda e^t \cdot M_Y(t), \quad M_Y'(0) = \lambda = E[Y] \;\checkmark
$$
$$
M_Y''(t) = \lambda e^t M_Y(t) + (\lambda e^t)^2 M_Y(t), \quad M_Y''(0) = \lambda + \lambda^2
$$
$$
\mathrm{Var}[Y] = (\lambda + \lambda^2) - \lambda^2 = \lambda \;\checkmark
$$

### Summary

$$
\boxed{E[Y] = \lambda, \quad \mathrm{Var}[Y] = \lambda, \quad v(\mu) = \mu, \quad g(\mu) = \ln(\mu) \text{ (log)}}
$$

**Note:** For the Poisson distribution $E[Y] = \mathrm{Var}[Y]$. When this equality does not hold in real data ("overdispersion" or "underdispersion"), one should consider quasi-Poisson or negative binomial models.

In [3]:
# --- Poisson: Monte Carlo verification ---
lam_true = 7.0
means, variances = [], []
for _ in range(n_sims):
    y = np.random.poisson(lam=lam_true, size=n_samples)
    means.append(np.mean(y))
    variances.append(np.var(y, ddof=1))

print("=== Poisson(lambda={}) ===".format(lam_true))
print(f"  Theoretical E[Y]   = {lam_true}")
print(f"  Simulated   E[Y]   = {np.mean(means):.4f}  (std over sims: {np.std(means):.4f})")
print(f"  Theoretical Var[Y] = {lam_true}")
print(f"  Simulated   Var[Y] = {np.mean(variances):.4f}  (std over sims: {np.std(variances):.4f})")
print(f"  v(mu)=mu  =>  Var[Y] / E[Y] = {np.mean(variances)/np.mean(means):.4f}  (should be ~1)")

=== Poisson(lambda=7.0) ===
  Theoretical E[Y]   = 7.0
  Simulated   E[Y]   = 6.9975  (std over sims: 0.0262)
  Theoretical Var[Y] = 7.0
  Simulated   Var[Y] = 6.9892  (std over sims: 0.1035)
  v(mu)=mu  =>  Var[Y] / E[Y] = 0.9988  (should be ~1)


---

## 3. Bernoulli Distribution $\mathrm{Ber}(p)$

This is the special case of $\mathrm{Binomial}(n=1, p)$.

**PMF:**
$$
f(y; p) = p^y (1-p)^{1-y}, \quad y \in \{0, 1\}
$$

### Step 1: Rewrite in exponential-family form

$$
f(y; p) = \exp\!\bigl[y \ln(p) + (1-y) \ln(1-p)\bigr]
= \exp\!\left[y \ln\!\left(\frac{p}{1-p}\right) + \ln(1-p)\right]
$$

### Step 2: Identify components

| Component | Value |
|-----------|-------|
| Natural parameter | $\theta = \ln\!\left(\frac{p}{1-p}\right)$ (log-odds), i.e. $p = \frac{e^\theta}{1 + e^\theta}$ |
| $b(\theta)$ | $\ln(1 + e^\theta)$ |
| $a(\phi)$ | $1$ |
| $c(y,\phi)$ | $0$ |

**Verification that $b(\theta) = \ln(1+e^\theta)$:** From $p = \frac{e^\theta}{1+e^\theta}$ we get $1-p = \frac{1}{1+e^\theta}$, so $\ln(1-p) = -\ln(1+e^\theta)$. Substituting back: $y\theta + \ln(1-p) = y\theta - \ln(1+e^\theta) = \frac{y\theta - b(\theta)}{1}$, confirming $b(\theta) = \ln(1+e^\theta)$.

### Step 3: Compute moments

$$
E[Y] = b'(\theta) = \frac{d}{d\theta}\ln(1+e^\theta) = \frac{e^\theta}{1 + e^\theta} = p
$$

For the second derivative, apply the quotient rule to $b'(\theta) = \frac{e^\theta}{1+e^\theta}$:
$$
b''(\theta) = \frac{e^\theta(1+e^\theta) - e^\theta \cdot e^\theta}{(1+e^\theta)^2} = \frac{e^\theta}{(1+e^\theta)^2}
$$

Recognizing that $\frac{e^\theta}{1+e^\theta} = p$ and $\frac{1}{1+e^\theta} = 1-p$:
$$
b''(\theta) = \frac{e^\theta}{1+e^\theta} \cdot \frac{1}{1+e^\theta} = p(1-p)
$$

$$
\mathrm{Var}[Y] = a(\phi)\,b''(\theta) = 1 \cdot p(1-p) = p(1-p)
$$

### Step 4: Variance function

$$
v(\mu) = b''(\theta(\mu)) = \mu(1-\mu)
$$

### Step 5: Canonical link

Since $\theta = \ln\!\left(\frac{p}{1-p}\right) = \ln\!\left(\frac{\mu}{1-\mu}\right)$, the canonical link is the **logit**: $g(\mu) = \ln\!\left(\frac{\mu}{1-\mu}\right)$.

### Summary

$$
\boxed{E[Y] = p, \quad \mathrm{Var}[Y] = p(1-p), \quad v(\mu) = \mu(1-\mu), \quad g(\mu) = \ln\!\left(\frac{\mu}{1-\mu}\right) \text{ (logit)}}
$$

In [4]:
# --- Bernoulli: Monte Carlo verification ---
p_true = 0.3
means, variances = [], []
for _ in range(n_sims):
    y = np.random.binomial(n=1, p=p_true, size=n_samples)
    means.append(np.mean(y))
    variances.append(np.var(y, ddof=1))

print("=== Bernoulli(p={}) ===".format(p_true))
print(f"  Theoretical E[Y]   = {p_true}")
print(f"  Simulated   E[Y]   = {np.mean(means):.4f}  (std over sims: {np.std(means):.4f})")
print(f"  Theoretical Var[Y] = {p_true*(1-p_true):.4f}")
print(f"  Simulated   Var[Y] = {np.mean(variances):.4f}  (std over sims: {np.std(variances):.4f})")
print(f"  v(mu)=mu(1-mu) = {p_true*(1-p_true):.4f}")

=== Bernoulli(p=0.3) ===
  Theoretical E[Y]   = 0.3
  Simulated   E[Y]   = 0.2995  (std over sims: 0.0042)
  Theoretical Var[Y] = 0.2100
  Simulated   Var[Y] = 0.2098  (std over sims: 0.0017)
  v(mu)=mu(1-mu) = 0.2100


---

## 4. Binomial Distribution $\mathrm{Bin}(n, p)$

**PMF:**
$$
P(Y = y) = \binom{n}{y} p^y (1-p)^{n-y}, \quad y = 0, 1, \ldots, n
$$

### Step 1: Rewrite in exponential-family form

$$
f(y; p) = \binom{n}{y} \exp\!\left[y \ln(p) + (n-y)\ln(1-p)\right]
= \exp\!\left[y \ln\!\left(\frac{p}{1-p}\right) + n\ln(1-p) + \ln\binom{n}{y}\right]
$$

### Step 2: Identify components

| Component | Value |
|-----------|-------|
| Natural parameter | $\theta = \ln\!\left(\frac{p}{1-p}\right)$, i.e. $p = \frac{e^\theta}{1+e^\theta}$ |
| $b(\theta)$ | $n\ln(1 + e^\theta)$ |
| $a(\phi)$ | $1$ |
| $c(y,\phi)$ | $-\ln\binom{n}{y}$ |

**Verification:** $y\theta - b(\theta) = y\ln\!\left(\frac{p}{1-p}\right) - n\ln(1+e^\theta) = y\ln\!\left(\frac{p}{1-p}\right) + n\ln(1-p)$, since $\ln(1+e^\theta) = -\ln(1-p)$. $\;\checkmark$

### Step 3: Compute moments

$$
E[Y] = b'(\theta) = n \cdot \frac{e^\theta}{1+e^\theta} = np
$$

$$
b''(\theta) = n \cdot \frac{e^\theta}{(1+e^\theta)^2} = np(1-p)
$$

(The derivative follows the same quotient rule as for Bernoulli, multiplied by $n$.)

$$
\mathrm{Var}[Y] = a(\phi)\,b''(\theta) = np(1-p)
$$

### Step 4: Variance function

Express in terms of $\mu = np$, so $p = \mu/n$:
$$
v(\mu) = np(1-p) = n \cdot \frac{\mu}{n} \cdot \left(1 - \frac{\mu}{n}\right) = \mu\left(1 - \frac{\mu}{n}\right)
$$

### Step 5: Canonical link

Same as Bernoulli: **logit** $g(p) = \ln\!\left(\frac{p}{1-p}\right)$.

Note: In practice, GLMs for binomial data model the proportion $p = \mu/n$, and the logit link is applied to $p$.

### Summary

$$
\boxed{E[Y] = np, \quad \mathrm{Var}[Y] = np(1-p), \quad v(\mu) = \mu\left(1 - \frac{\mu}{n}\right), \quad g(p) = \mathrm{logit}(p)}
$$

In [5]:
# --- Binomial: Monte Carlo verification ---
n_binom, p_true = 20, 0.4
means, variances = [], []
for _ in range(n_sims):
    y = np.random.binomial(n=n_binom, p=p_true, size=n_samples)
    means.append(np.mean(y))
    variances.append(np.var(y, ddof=1))

mu_true = n_binom * p_true
var_true = n_binom * p_true * (1 - p_true)

print("=== Binomial(n={}, p={}) ===".format(n_binom, p_true))
print(f"  Theoretical E[Y]   = {mu_true}")
print(f"  Simulated   E[Y]   = {np.mean(means):.4f}  (std over sims: {np.std(means):.4f})")
print(f"  Theoretical Var[Y] = {var_true:.4f}")
print(f"  Simulated   Var[Y] = {np.mean(variances):.4f}  (std over sims: {np.std(variances):.4f})")
print(f"  v(mu)=mu(1-mu/n) = {mu_true*(1 - mu_true/n_binom):.4f}")

=== Binomial(n=20, p=0.4) ===
  Theoretical E[Y]   = 8.0
  Simulated   E[Y]   = 8.0010  (std over sims: 0.0231)
  Theoretical Var[Y] = 4.8000
  Simulated   Var[Y] = 4.7900  (std over sims: 0.0615)
  v(mu)=mu(1-mu/n) = 4.8000


---

## 5. Gamma Distribution $\Gamma(p, a)$

With shape $p > 0$ and rate $a > 0$.

**PDF:**
$$
f(y; a, p) = \frac{a^p}{\Gamma(p)} y^{p-1} e^{-ay}, \quad y > 0
$$

### Step 1: Rewrite in exponential-family form

Take the logarithm:
$$
\ln f(y; a, p) = -ay + (p-1)\ln(y) + p\ln(a) - \ln\Gamma(p)
$$

The mean is $\mu = p/a$, so $a = p/\mu$. In the GLM convention, the shape $p$ is treated as known and we set the dispersion $\phi = 1/p$. The natural parameter is $\theta = -a/p = -1/\mu$ (so $\theta < 0$ and $\mu = -1/\theta$).

Rewriting in terms of $\theta$ (where $a = -p\theta$ and $\ln a = \ln p + \ln(-\theta)$):
$$
\ln f = p\theta y + (p-1)\ln(y) + p\ln(p) + p\ln(-\theta) - \ln\Gamma(p)
$$

Separating the $y\theta$ term and grouping:
$$
\ln f = \frac{\theta y - (-\ln(-\theta))}{1/p} + (p-1)\ln(y) + p\ln(p) - \ln\Gamma(p)
$$

since $p \cdot [\theta y + \ln(-\theta)] = \frac{\theta y - (-\ln(-\theta))}{1/p}$.

### Step 2: Identify components

| Component | Value |
|-----------|-------|
| Natural parameter | $\theta = -1/\mu = -a/p < 0$ |
| $b(\theta)$ | $-\ln(-\theta)$ |
| $a(\phi)$ | $1/p$ (shape $p$ plays the role of dispersion) |
| $c(y,\phi)$ | $-(p-1)\ln(y) - p\ln(p) + \ln\Gamma(p)$ |

### Step 3: Compute moments

Using chain rule with $u = -\theta > 0$:
$$
b'(\theta) = \frac{d}{d\theta}\bigl[-\ln(-\theta)\bigr] = \frac{d}{d\theta}[-\ln u] = -\frac{1}{u}\cdot\frac{du}{d\theta} = -\frac{1}{(-\theta)}\cdot(-1) = \frac{1}{-\theta} = -\frac{1}{\theta}
$$

Since $\theta = -1/\mu$:
$$
E[Y] = b'(\theta) = -\frac{1}{\theta} = -\frac{1}{-1/\mu} = \mu = \frac{p}{a} \;\checkmark
$$

$$
b''(\theta) = \frac{d}{d\theta}\left(-\frac{1}{\theta}\right) = \frac{1}{\theta^2}
$$

Since $\theta = -1/\mu$, we get $\theta^2 = 1/\mu^2$, so:
$$
b''(\theta) = \mu^2
$$

$$
\mathrm{Var}[Y] = a(\phi)\,b''(\theta) = \frac{1}{p} \cdot \mu^2 = \frac{\mu^2}{p} = \frac{(p/a)^2}{p} = \frac{p}{a^2} \;\checkmark
$$

### Step 4: Variance function

$$
v(\mu) = b''(\theta(\mu)) = \mu^2
$$

### Step 5: Canonical link

Since $\theta = -1/\mu$, the canonical link is the **reciprocal** (inverse): $g(\mu) = 1/\mu$.

(The sign convention varies across texts; many define $g(\mu) = 1/\mu$ and absorb the negative sign into the parameterization.)

### Verification via MGF

The MGF of $Y \sim \Gamma(p, a)$ is:
$$
M_Y(t) = \left(\frac{a}{a-t}\right)^p = \left(1 - \frac{t}{a}\right)^{-p}, \quad t < a
$$

$$
M_Y'(t) = p \cdot \frac{1}{a-t} \cdot \left(\frac{a}{a-t}\right)^p, \quad M_Y'(0) = \frac{p}{a} = \mu \;\checkmark
$$

$$
M_Y''(0) = \frac{p(p+1)}{a^2}
$$

$$
\mathrm{Var}[Y] = \frac{p(p+1)}{a^2} - \left(\frac{p}{a}\right)^2 = \frac{p^2 + p - p^2}{a^2} = \frac{p}{a^2} \;\checkmark
$$

### Summary

$$
\boxed{E[Y] = \frac{p}{a}, \quad \mathrm{Var}[Y] = \frac{p}{a^2}, \quad v(\mu) = \mu^2, \quad g(\mu) = \frac{1}{\mu} \text{ (reciprocal)}}
$$

**Note:** The Gamma distribution is commonly used for modeling positive, right-skewed data (e.g. insurance claims, waiting times). The variance grows with $\mu^2$, which makes it suitable for data where variability increases proportionally to the mean.

In [6]:
# --- Gamma: Monte Carlo verification ---
# Gamma(shape=p, rate=a): E[Y]=p/a, Var[Y]=p/a^2
p_shape, a_rate = 3.0, 2.0
mu_true = p_shape / a_rate
var_true = p_shape / a_rate**2
means, variances = [], []
for _ in range(n_sims):
    # numpy uses shape and scale=1/rate
    y = np.random.gamma(shape=p_shape, scale=1.0/a_rate, size=n_samples)
    means.append(np.mean(y))
    variances.append(np.var(y, ddof=1))

print("=== Gamma(shape={}, rate={}) ===".format(p_shape, a_rate))
print(f"  Theoretical E[Y]   = {mu_true:.4f}")
print(f"  Simulated   E[Y]   = {np.mean(means):.4f}  (std over sims: {np.std(means):.4f})")
print(f"  Theoretical Var[Y] = {var_true:.4f}")
print(f"  Simulated   Var[Y] = {np.mean(variances):.4f}  (std over sims: {np.std(variances):.4f})")
print(f"  v(mu)=mu^2 = {mu_true**2:.4f},  Var/a(phi) = {np.mean(variances) * p_shape:.4f}  (should be ~{mu_true**2:.4f})")

=== Gamma(shape=3.0, rate=2.0) ===
  Theoretical E[Y]   = 1.5000
  Simulated   E[Y]   = 1.5007  (std over sims: 0.0078)
  Theoretical Var[Y] = 0.7500
  Simulated   Var[Y] = 0.7498  (std over sims: 0.0132)
  v(mu)=mu^2 = 2.2500,  Var/a(phi) = 2.2495  (should be ~2.2500)


---

## 6. Inverse Gaussian Distribution $\mathrm{IG}(\mu, \lambda)$

**PDF:**
$$
f(y; \mu, \lambda) = \sqrt{\frac{\lambda}{2\pi y^3}} \exp\!\left[-\frac{\lambda(y-\mu)^2}{2\mu^2 y}\right], \quad y > 0
$$

The Inverse Gaussian distribution arises naturally as the distribution of the first passage time of a Brownian motion with positive drift.

### Step 1: Rewrite in exponential-family form

Expand the exponent:
$$
-\frac{\lambda(y-\mu)^2}{2\mu^2 y} = -\frac{\lambda(y^2 - 2y\mu + \mu^2)}{2\mu^2 y}
= -\frac{\lambda y}{2\mu^2} + \frac{\lambda}{\mu} - \frac{\lambda}{2y}
$$

Collecting all terms (including the square root prefactor via $\sqrt{\frac{\lambda}{2\pi y^3}} = \exp\!\left[\frac{1}{2}\ln\lambda - \frac{3}{2}\ln y - \frac{1}{2}\ln(2\pi)\right]$):

$$
\ln f(y; \mu, \lambda) = -\frac{\lambda y}{2\mu^2} + \frac{\lambda}{\mu} - \frac{\lambda}{2y} + \frac{1}{2}\ln\lambda - \frac{3}{2}\ln y - \frac{1}{2}\ln(2\pi)
$$

Rearranging with $a(\phi) = 1/\lambda$:

$$
= \frac{-\frac{1}{2\mu^2} y - \left(-\frac{1}{\mu}\right)}{1/\lambda} + \underbrace{\left(-\frac{\lambda}{2y} + \frac{1}{2}\ln\lambda - \frac{3}{2}\ln y - \frac{1}{2}\ln(2\pi)\right)}_{-c(y,\phi)}
$$

### Step 2: Identify components

| Component | Value |
|-----------|-------|
| Natural parameter | $\theta = -\frac{1}{2\mu^2}$ (note: $\theta < 0$) |
| $b(\theta)$ | $-\frac{1}{\mu} = -\sqrt{-2\theta}$ |
| $a(\phi)$ | $1/\lambda$ |
| $c(y,\phi)$ | $\frac{\lambda}{2y} - \frac{1}{2}\ln\lambda + \frac{3}{2}\ln y + \frac{1}{2}\ln(2\pi)$ |

**Verification:** $\frac{y\theta - b(\theta)}{a(\phi)} = \lambda\!\left(-\frac{y}{2\mu^2} + \frac{1}{\mu}\right) = \frac{\lambda}{\mu} - \frac{\lambda y}{2\mu^2}$ $\;\checkmark$

### Step 3: Compute moments

From $b(\theta) = -(-2\theta)^{1/2}$, apply the chain rule with $u = -2\theta > 0$:

$$
b'(\theta) = -\frac{1}{2} u^{-1/2} \cdot \frac{du}{d\theta} = -\frac{1}{2}(-2\theta)^{-1/2} \cdot (-2) = (-2\theta)^{-1/2} = \frac{1}{\sqrt{-2\theta}}
$$

Since $\theta = -\frac{1}{2\mu^2}$, we get $-2\theta = \frac{1}{\mu^2}$ and $\sqrt{-2\theta} = \frac{1}{\mu}$:
$$
E[Y] = b'(\theta) = \frac{1}{1/\mu} = \mu \;\checkmark
$$

For the second derivative, differentiate $b'(\theta) = (-2\theta)^{-1/2}$:
$$
b''(\theta) = -\frac{1}{2}(-2\theta)^{-3/2} \cdot (-2) = (-2\theta)^{-3/2}
$$

Since $(-2\theta)^{-3/2} = \left(\frac{1}{\mu^2}\right)^{-3/2} = (\mu^2)^{3/2} = \mu^3$:
$$
b''(\theta) = \mu^3
$$

$$
\mathrm{Var}[Y] = a(\phi)\,b''(\theta) = \frac{1}{\lambda} \cdot \mu^3 = \frac{\mu^3}{\lambda}
$$

### Step 4: Variance function

$$
v(\mu) = b''(\theta(\mu)) = \mu^3
$$

### Step 5: Canonical link

Since $\theta = -\frac{1}{2\mu^2}$, the canonical link is $g(\mu) = \frac{1}{\mu^2}$.

(The negative sign and factor of $1/2$ are absorbed into the parameterization.)

### Summary

$$
\boxed{E[Y] = \mu, \quad \mathrm{Var}[Y] = \frac{\mu^3}{\lambda}, \quad v(\mu) = \mu^3, \quad g(\mu) = \frac{1}{\mu^2}}
$$

**Note:** The Inverse Gaussian has the fastest-growing variance function among the Big Five ($v(\mu) = \mu^3$), making it suitable for data with high variability at large mean values.

In [7]:
# --- Inverse Gaussian: Monte Carlo verification ---
# IG(mu, lambda): E[Y]=mu, Var[Y]=mu^3/lambda
mu_true, lam_true = 3.0, 5.0
var_true = mu_true**3 / lam_true
means, variances = [], []
for _ in range(n_sims):
    y = np.random.wald(mean=mu_true, scale=lam_true, size=n_samples)
    means.append(np.mean(y))
    variances.append(np.var(y, ddof=1))

print("=== Inverse Gaussian(mu={}, lambda={}) ===".format(mu_true, lam_true))
print(f"  Theoretical E[Y]   = {mu_true:.4f}")
print(f"  Simulated   E[Y]   = {np.mean(means):.4f}  (std over sims: {np.std(means):.4f})")
print(f"  Theoretical Var[Y] = {var_true:.4f}")
print(f"  Simulated   Var[Y] = {np.mean(variances):.4f}  (std over sims: {np.std(variances):.4f})")
print(f"  v(mu)=mu^3 = {mu_true**3:.4f}")

=== Inverse Gaussian(mu=3.0, lambda=5.0) ===
  Theoretical E[Y]   = 3.0000
  Simulated   E[Y]   = 2.9977  (std over sims: 0.0224)
  Theoretical Var[Y] = 5.4000
  Simulated   Var[Y] = 5.4116  (std over sims: 0.1676)
  v(mu)=mu^3 = 27.0000


---

## Summary Table

| Distribution | $E[Y]$ | $\mathrm{Var}[Y]$ | $v(\mu)$ | Canonical link $g(\mu)$ |
|:---|:---|:---|:---|:---|
| Normal $N(\mu,\sigma^2)$ | $\mu$ | $\sigma^2$ | $1$ | $\mu$ (identity) |
| Poisson $\mathrm{Pois}(\lambda)$ | $\lambda$ | $\lambda$ | $\mu$ | $\ln(\mu)$ (log) |
| Bernoulli $\mathrm{Ber}(p)$ | $p$ | $p(1-p)$ | $\mu(1-\mu)$ | $\ln\frac{\mu}{1-\mu}$ (logit) |
| Binomial $\mathrm{Bin}(n,p)$ | $np$ | $np(1-p)$ | $\mu(1-\mu/n)$ | logit |
| Gamma $\Gamma(p,a)$ | $p/a$ | $p/a^2$ | $\mu^2$ | $1/\mu$ (reciprocal) |
| Inverse Gaussian $\mathrm{IG}(\mu,\lambda)$ | $\mu$ | $\mu^3/\lambda$ | $\mu^3$ | $1/\mu^2$ |

**Power family pattern:** Notice that the variance functions form a power family $v(\mu) = \mu^k$ with $k = 0$ (Normal), $k = 1$ (Poisson), $k = 2$ (Gamma), $k = 3$ (Inverse Gaussian). The Bernoulli/Binomial case $v(\mu) = \mu(1-\mu)$ does not fit this pattern due to the bounded support $[0,1]$ (or $[0,n]$).

---

## Answers to HW questions

**Q: Which distributions can fulfill homoscedasticity and why?**

Only the **Normal** distribution, because it is the only one with variance function $v(\mu) = 1$, meaning $\mathrm{Var}[Y]$ does not depend on $\mu$.

**Q: For which distribution does the variance increase with the square of the mean?**

The **Gamma** distribution, where $v(\mu) = \mu^2$, so $\mathrm{Var}[Y] = \phi \cdot \mu^2$.

**Q: Does there exist a distribution where $V[Y] = k \cdot \mu$?**

Yes, the **Poisson** distribution with $v(\mu) = \mu$ and $a(\phi) = 1$, giving $\mathrm{Var}[Y] = \mu$. More generally, any Poisson-like model with overdispersion parameter gives $\mathrm{Var}[Y] = \phi \cdot \mu$.

---

## Note on the MGF approach

For any exponential-family distribution, the moments can be computed via the moment generating function:
$$
M_Y(t) = E[e^{tY}], \quad E[Y] = M_Y'(0), \quad \mathrm{Var}[Y] = M_Y''(0) - (M_Y'(0))^2
$$

Alternatively, using the exponential-family structure directly:
$$
E[Y] = b'(\theta), \quad \mathrm{Var}[Y] = a(\phi)\,b''(\theta)
$$

Both approaches must yield the same result — this serves as a useful cross-check.